# Admission Check: Published Optimum Reproduction

![Python](https://img.shields.io/badge/python-3.10%2B-blue) ![Status](https://img.shields.io/badge/status-research--reproduction-lightgrey) ![License](https://img.shields.io/badge/license-MIT-green)

Confirms that each problem implementation reproduces its published best-known objective value at the published solution point, within tolerance and near-feasibility. Problems that fail this check would be excluded from the study.


## Imports

External libraries and project modules used by this notebook.

In [ ]:
import numpy as np
from problems import PROBLEMS

In [ ]:
FEAS_TOL = 1e-4
REL_TOL = 5e-3

rows = []
for p in PROBLEMS:
    x = p.known_x.reshape(1, -1).astype(float)
    f, G = p.evaluate(x)
    fv = float(f[0])
    viol = float(np.maximum(G[0], 0.0).sum())
    rel = abs(fv - p.known_f) / max(abs(p.known_f), 1e-12)
    ok_f = rel <= REL_TOL
    ok_g = viol <= FEAS_TOL
    rows.append((p.name, p.dim, p.n_con, p.known_f, fv, rel, viol, ok_f and ok_g))

print(f"{'problem':<18}{'D':>3}{'m':>4}{'published f':>16}{'computed f':>16}"
      f"{'rel.err':>12}{'violation':>14}  status")
print("-" * 96)
for name, d, m, kf, fv, rel, viol, ok in rows:
    print(f"{name:<18}{d:>3}{m:>4}{kf:>16.7g}{fv:>16.7g}{rel:>12.3e}{viol:>14.3e}"
          f"  {'PASS' if ok else 'FAIL'}")

n_pass = sum(r[-1] for r in rows)
print("-" * 96)
print(f"{n_pass}/{len(rows)} problems verified")

---
## Key outcomes

- This admission check is a hard gate: a problem only enters the study if its published best-known
  solution reproduces the published objective value (within 0.5% relative error) and is feasible
  (constraint violation <= 1e-4).
- 8 of the 9 problems in `PROBLEMS` pass; **DiscBrake** is subsequently dropped from all reported
  comparisons (see notebook 01) despite technically passing this specific check, because a separate
  search-based test shows algorithms readily beating its published optimum.

*Part of the budget-controlled reproduction study of nature-inspired metaheuristics (Engineering Research Express).*
